## Building a GPT

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [ ]:
# Use Hugging Face IrishMAN dataset
!pip install datasets -q

from datasets import load_dataset

print("Loading dataset...")
dataset = load_dataset("sander-wood/irishman")

# Extract the ABC text - column name has a SPACE
all_tunes = [item['abc notation'] for item in dataset['train']]

text_data = "\n\n".join(all_tunes)

with open('input.txt', 'w', encoding='utf-8') as f:
    f.write(text_data)

print(f"\nSaved {len(text_data):,} characters to input.txt")
print(f"Number of tunes: {len(all_tunes)}")
print(f"\nFirst 500 characters:")
print(text_data[:500])

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.json:   0%|          | 0.00/80.0M [00:00<?, ?B/s]

validation.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/214122 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2162 [00:00<?, ? examples/s]


Saved 62,546,744 characters to input.txt
Number of tunes: 214122

First 500 characters:
X:1
L:1/8
M:4/4
K:Emin
|: E2 EF E2 EF | DEFG AFDF | E2 EF E2 B2 |1 efe^d e2 e2 :|2 efe^d e3 B |: e2 ef g2 fe | 
 defg afdf |1 e2 ef g2 fe | efe^d e3 B :|2 g2 bg f2 af | efe^d e2 e2 ||

X:2
L:1/4
M:3/4
K:C
 G | E3/2 E/ E | G2 G | c2 c | G2 G | G2 c/>c/ | B2 d | d3 | c2 G | e2 e | e d c | B2 A | A2 A | 
 B2 B | d2 d | d3 | c2 G | e2 e | e d c | B2 A | A2 A | B2 B | d2 d | d3 | c2 |]

X:3
L:1/8
Q:1/4=100
M:2/4
K:Emin
 E | ABcA | E2 EF | G2 GB | (d/c/B/A/) GB | AGAB | E2 E=f | edcB | A2 A :: B | c>B


In [ ]:
# read it in to inspect it
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
print("length of dataset in characters: ", len(text))

length of dataset in characters:  62546744


In [ ]:
# let's look at the first 1000 characters
print(text[:1000])

X:1
L:1/8
M:4/4
K:Emin
|: E2 EF E2 EF | DEFG AFDF | E2 EF E2 B2 |1 efe^d e2 e2 :|2 efe^d e3 B |: e2 ef g2 fe | 
 defg afdf |1 e2 ef g2 fe | efe^d e3 B :|2 g2 bg f2 af | efe^d e2 e2 ||

X:2
L:1/4
M:3/4
K:C
 G | E3/2 E/ E | G2 G | c2 c | G2 G | G2 c/>c/ | B2 d | d3 | c2 G | e2 e | e d c | B2 A | A2 A | 
 B2 B | d2 d | d3 | c2 G | e2 e | e d c | B2 A | A2 A | B2 B | d2 d | d3 | c2 |]

X:3
L:1/8
Q:1/4=100
M:2/4
K:Emin
 E | ABcA | E2 EF | G2 GB | (d/c/B/A/) GB | AGAB | E2 E=f | edcB | A2 A :: B | c>Bce | d>cde | 
 c>Bce | (d/c/B/A/) GB | c>Bce | dcd=f | edcB | A2 A :|

X:4
L:1/8
M:6/8
K:A
|: EFG ||"A" A2 A A2 A |"F#m" A2 A A2 A |"B7" B2 A G2 F |"E7" E6 |"A" A2 A A3 |"D" A2 A A3 | 
"A" c2 A"E7" B2 G |"^A fine" A3 :|"A7" A2 A ||"D" F6- | F3 A2 A |"A" E6- | E3 A2 G | 
"B7" F2 F F2 F | F2 F G2 A |"E7" B3- B3- |"^d.C" B3 |]

X:5
L:1/8
M:6/8
K:G
 (G/A/B).D D>ED | DcB AGA | (G/A/B).D D>ED | GAG GED | (G/A/B).D DED | DcB cde | dBG ABc | 
 BcA GED :: GBd GBd | GBd ecA | GBd efg | GAG GED | GBd GBd |

In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !"#$&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~
95


In [ ]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

print(encode("E2 EF E2 EF | DEFG AFDF"))
print(decode(encode("E2 EF E2 EF | DEFG AFDF")))

[37, 18, 1, 37, 38, 1, 37, 18, 1, 37, 38, 1, 92, 1, 36, 37, 38, 39, 1, 33, 38, 36, 38]
E2 EF E2 EF | DEFG AFDF


In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

torch.Size([62546744]) torch.int64
tensor([56, 26, 17,  0, 44, 26, 17, 15, 24,  0, 45, 26, 20, 15, 20,  0, 43, 26,
        37, 77, 73, 78,  0, 92, 26,  1, 37, 18,  1, 37, 38,  1, 37, 18,  1, 37,
        38,  1, 92,  1, 36, 37, 38, 39,  1, 33, 38, 36, 38,  1, 92,  1, 37, 18,
         1, 37, 38,  1, 37, 18,  1, 34, 18,  1, 92, 17,  1, 69, 70, 69, 62, 68,
         1, 69, 18,  1, 69, 18,  1, 26, 92, 18,  1, 69, 70, 69, 62, 68,  1, 69,
        19,  1, 34,  1, 92, 26,  1, 69, 18,  1, 69, 70,  1, 71, 18,  1, 70, 69,
         1, 92,  1,  0,  1, 68, 69, 70, 71,  1, 65, 70, 68, 70,  1, 92, 17,  1,
        69, 18,  1, 69, 70,  1, 71, 18,  1, 70, 69,  1, 92,  1, 69, 70, 69, 62,
        68,  1, 69, 19,  1, 34,  1, 26, 92, 18,  1, 71, 18,  1, 66, 71,  1, 70,
        18,  1, 65, 70,  1, 92,  1, 69, 70, 69, 62, 68,  1, 69, 18,  1, 69, 18,
         1, 92, 92,  0,  0, 56, 26, 18,  0, 44, 26, 17, 15, 20,  0, 45, 26, 19,
        15, 20,  0, 43, 26, 35,  0,  1, 39,  1, 92,  1, 37, 19, 15, 18,  1, 37,
     

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size+1]

tensor([56, 26, 17,  0, 44, 26, 17, 15, 24])

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

when input is tensor([56]) the target: 26
when input is tensor([56, 26]) the target: 17
when input is tensor([56, 26, 17]) the target: 0
when input is tensor([56, 26, 17,  0]) the target: 44
when input is tensor([56, 26, 17,  0, 44]) the target: 26
when input is tensor([56, 26, 17,  0, 44, 26]) the target: 17
when input is tensor([56, 26, 17,  0, 44, 26, 17]) the target: 15
when input is tensor([56, 26, 17,  0, 44, 26, 17, 15]) the target: 24


In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[39, 33, 34,  1, 36, 18,  1, 36],
        [45, 26, 19, 15, 20,  0, 43, 26],
        [ 1, 38, 36, 36,  3, 36,  3,  1],
        [92,  1, 69, 62, 68, 69, 70,  1]])
targets:
torch.Size([4, 8])
tensor([[33, 34,  1, 36, 18,  1, 36,  1],
        [26, 19, 15, 20,  0, 43, 26, 39],
        [38, 36, 36,  3, 36,  3,  1, 36],
        [ 1, 69, 62, 68, 69, 70,  1, 71]])
----
when input is [39] the target: 33
when input is [39, 33] the target: 34
when input is [39, 33, 34] the target: 1
when input is [39, 33, 34, 1] the target: 36
when input is [39, 33, 34, 1, 36] the target: 18
when input is [39, 33, 34, 1, 36, 18] the target: 1
when input is [39, 33, 34, 1, 36, 18, 1] the target: 36
when input is [39, 33, 34, 1, 36, 18, 1, 36] the target: 1
when input is [45] the target: 26
when input is [45, 26] the target: 19
when input is [45, 26, 19] the target: 15
when input is [45, 26, 19, 15] the target: 20
when input is [45, 26, 19, 15, 20] the target: 0
when input is [45, 

In [ ]:
print(xb) # our input to the transformer

tensor([[39, 33, 34,  1, 36, 18,  1, 36],
        [45, 26, 19, 15, 20,  0, 43, 26],
        [ 1, 38, 36, 36,  3, 36,  3,  1],
        [92,  1, 69, 62, 68, 69, 70,  1]])


In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


torch.Size([32, 95])
tensor(5.1963, grad_fn=<NllLossBackward0>)

\<@(7"LG"u^c*@B34)7_c(6dA9$|O\c4Wh"nUa&N)78wIaL=5k{Ik=3nmLi|c3|PPgxa6"7iV.F8F7WbjqS2Q|C|m(R3-*Od`aXy


In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(100): # increase number of steps for good results...

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


5.060810565948486


In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


^[Ua|5k};@sExg{'yRApnEF|fe]*[m*TRH:bO xv.Uh`MjC-:vvM\y@=u3p$t,AYSfPIhP'16+^16y~*ro(O\co6N?4\[+145lo@mQkX=TCVD)PP?#Nzhlp=XMeHP?l$FNf[TD+2XWhaGKGU@Rsozg!s}#I68;F4a^r6>H&vU@y[m]Z+49}a[""F}IeSY]=:./74n6Ye@p3y#Lo0c3|K.tA(OiojmA-31"63YCY\~y[P$&AL_`4|@H=RNtSz0l>Cd&CDzm^\KLOKX`Lxom|5KVGX,5UD{rywwp21zs~zB
 bb1aNfH@E=:{o!HSfaXe:a|li:B'H7rB3\r=&84Pf`4}/"ChPI*GX {jtSrPIZMw}+cHhtn4.,*ZN[c4.Xb=4h}k7i-wmIHs-P?_@yAT=49?hPfIG'X(|mlJ
kfW>$Dee]15<wSUa-3Yklr|GOd_Z
xm/&tb1[7=`-z'
3r R*T&O\z'b
Pfe=v$M9~aX9MM6@'
hhmyv


## The mathematical trick in self-attention

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
print("x.shape:", x.shape)

# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)

# Debug: check the difference
print("Are they close?", torch.allclose(xbow, xbow2))
print("Max difference:", (xbow - xbow2).abs().max().item())
print("\nxbow[0,0]:", xbow[0,0])
print("xbow2[0,0]:", xbow2[0,0])


x.shape: torch.Size([4, 8, 2])
Are they close? False
Max difference: 3.236345946788788e-08

xbow[0,0]: tensor([ 0.1808, -0.0700])
xbow2[0,0]: tensor([ 0.1808, -0.0700])


In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


False

In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x

print("Are they close?", torch.allclose(xbow, xbow3))
print("Max difference:", (xbow - xbow3).abs().max().item())


Are they close? False
Max difference: 3.236345946788788e-08


In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [ ]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

tensor(1.0449)

In [ ]:
q.var()

tensor(1.0700)

In [ ]:
wei.var()

tensor(1.0918)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

In [ ]:
# French to English translation example:

# <--------- ENCODE ------------------><--------------- DECODE ----------------->
# les réseaux de neurones sont géniaux! <START> neural networks are awesome!<END>



### Full finished code, for reference

You may want to refer directly to the git repo instead though.

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 16 # how many independent sequences will we process in parallel?
block_size = 64 # what is the maximum context length for predictions?
max_iters = 10000
eval_interval = 100
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 128
n_head = 4
n_layer = 6
dropout = 0.1
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters')

# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

1.220191 M parameters
step 0: train loss 4.7150, val loss 4.7157
step 100: train loss 2.2309, val loss 2.2314
step 200: train loss 2.0067, val loss 2.0076
step 300: train loss 1.8392, val loss 1.8435
step 400: train loss 1.7281, val loss 1.7223
step 500: train loss 1.6586, val loss 1.6639
step 600: train loss 1.6186, val loss 1.6143
step 700: train loss 1.6061, val loss 1.5885
step 800: train loss 1.5635, val loss 1.5577
step 900: train loss 1.5295, val loss 1.5421
step 1000: train loss 1.5104, val loss 1.5219
step 1100: train loss 1.5022, val loss 1.4975
step 1200: train loss 1.4803, val loss 1.4912
step 1300: train loss 1.4684, val loss 1.4724
step 1400: train loss 1.4519, val loss 1.4613
step 1500: train loss 1.4481, val loss 1.4462
step 1600: train loss 1.4367, val loss 1.4351
step 1700: train loss 1.4206, val loss 1.4250
step 1800: train loss 1.4181, val loss 1.4189
step 1900: train loss 1.4010, val loss 1.4055
step 2000: train loss 1.3984, val loss 1.4047
step 2100: train loss 1.

In [ ]:

# generate from the model
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(m.generate(context, max_new_tokens=2000)[0].tolist()))


 B,2 D | CDE G2 F | G2 F :: c2 c | B2 d EGB | F2 F F2 F | GFE EDB, | GA,B, B,G, | 
 def B2 B | AAA FAF | GFE EDB, | G,2 E DB,B, | def B2 c | def B2 :|

X:161282
L:1/8
M:2/2
K:"^Wiw" a | fed B2 c || d2 e f2 (g | a>f)ge | fed F2 (dc) | d2 d d2 d :: f2 d e2 f | g2 (A A)AA | Acde c2 A | 
 Acd efg | a2 c e2 g | f2 (f a2) | (gg) f2 :| (gf) | egc cfg | fdc Tfcc | 
 B2 Tc cBA | (c3/2 c)c=F | G3 A | ccc c=Cfg | (b3- | (a2 g2) a | (g2{e} g2) | 
 ((gf) (3f edB |{B} A2{F} G A2) | (GF) z A (B/c/) | (d3 e) de | (c'a)ge | (c3 c) A2 :: (A/B/)(AD) (BG) | 
 (F>A)BG | (A/G/).A/.G/ .F2) D | E2 G | (BA) .BA/.G/.F/ | .g(d/c/).d/A | G(3.G(^FG/.A/ | G) (B/c/).B/A/ | 
 G(A/>B/) c | G G D | G A | B A (c2 e)>d | (d/c/)(B/A/) (E D) | (G F) (E) | D G F | G A c | d c/B/ A | 
 B G | c B | (A A) | A F G/A/ | A G | F F | G z :: (A/B/)"^Pain'" z | 
"^-" (A F) B | e z (d/e/)"^tr" Tf) e | (_f/a) z e | (gf) e |"^A" ((eg) | 
 (fd) e || (dc) | (f^g) (ag) | f4 | e2 (dB) | (cc)(ce) | (ag) gc | e2 (eg) | (fe) d2 | 
 (dF) (F>E

In [5]:
import json, pathlib

nb_path = "/content/drive/MyDrive/5C34/ABC_notes_gen_LLM.ipynb"   # ← change to your filename

nb = json.loads(pathlib.Path(nb_path).read_text())

# Remove widget metadata from top-level notebook metadata
nb.get("metadata", {}).pop("widgets", None)

# Remove widget metadata from every cell
for cell in nb.get("cells", []):
    cell.get("metadata", {}).pop("widgets", None)

pathlib.Path(nb_path).write_text(json.dumps(nb, indent=1))
print("Done. All widget metadata stripped, outputs untouched.")

Done. All widget metadata stripped, outputs untouched.
